# Aula 4 - MCP (Model Context Protocol)

## Versão 4.0 do projeto: três arquivos, três responsabilidades

Até a Aula 3, tudo vivia em um único notebook: as ferramentas eram funções Python locais, chamadas diretamente pela aplicação. A partir de agora, o projeto se divide em três arquivos, cada um com uma responsabilidade clara:

- **`MCPServer.py`** — o servidor MCP. Expõe **tools** (ações), **resources** (dados) e **prompts** (templates), sem saber nada sobre qual modelo de IA vai usá-los.
- **`MCPClient.py`** — a ponte. Sobe o `MCPServer.py` como um subprocesso e abre uma conexão MCP via STDIO com ele.
- **`Main.py`** — a aplicação. Conecta-se ao servidor através do client, descobre o que ele oferece, conversa com o modelo e executa as tools que o modelo pedir.

Este notebook te guia na construção dos três arquivos, nessa ordem, usando a célula mágica `%%writefile` para efetivamente criar cada `.py` na pasta do projeto. No fim, os TODOs interativos do notebook viram a base para o `Main.py` final, que você vai rodar pelo terminal.

Ao final, você deve conseguir:

- criar um MCP Server com `@mcp.tool()`, `@mcp.resource()` e `@mcp.prompt()`;
- criar um MCP Client que sobe esse servidor via STDIO e devolve uma sessão utilizável;
- descobrir dinamicamente tools, resources e prompts a partir dessa sessão;
- converter tools MCP para o formato de function calling da API;
- validar o nome de uma tool antes de executá-la, mesmo vindo de um servidor externo;
- executar uma tool remota com `session.call_tool` e devolver o resultado ao modelo;
- consolidar tudo em um `Main.py` executável pelo terminal.

## 1. Configuração do ambiente

Instale as dependências pelo terminal, seguindo o `README.md` desta pasta (agora incluindo o pacote `mcp`). Copie `env.example` para `.env` e preencha `GROQ_API_KEY`. Nunca coloque a chave diretamente neste arquivo nem faça commit do `.env`.

As próximas células vão criar arquivos `.py` na mesma pasta deste notebook. Rode-as em ordem: cada arquivo depende do anterior.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

## 2. Construa o `MCPServer.py`

O servidor é a parte que "conhece" o domínio (aulas e notas), mas não sabe nada sobre LLMs, prompts de sistema ou o Groq. Ele só declara, via decoradores, o que está disponível para quem se conectar:

- `@mcp.tool()` — uma função que pode ser executada sob pedido;
- `@mcp.resource("uri")` — dados que podem ser lidos por uma URI;
- `@mcp.prompt()` — um template de prompt parametrizável.

A célula abaixo usa `%%writefile` para gravar `MCPServer.py` de verdade nesta pasta. Complete os TODOs e rode a célula (rodar de novo sobrescreve o arquivo).

In [ ]:
%%writefile MCPServer.py
import json

from mcp.server import MCPServer

mcp = MCPServer("aula4-tools")

STUDY_CONTENT = {
    "1": {"tema": "LLM APIs", "resumo": "Chamadas, mensagens, tokens e ausência de memória automática."},
    "2": {"tema": "Context Engineering", "resumo": "Seleção e organização do contexto antes da chamada."},
    "3": {"tema": "Tool Calling", "resumo": "Contratos de ferramentas, pedidos estruturados e execução pela aplicação."},
}

# TODO: implemente `consultar_aula(aula: str) -> dict`, decorada com `@mcp.tool()`.
# Se `aula` não estiver em STUDY_CONTENT, devolva um dicionário com "erro" e "aulas_disponiveis"
# (a lista das chaves de STUDY_CONTENT). Caso contrário, devolva um dicionário com "aula"
# e os dados de STUDY_CONTENT[aula] (tema e resumo).


# TODO: implemente `calcular_media(notas: list[float]) -> dict`, decorada com `@mcp.tool()`.
# Se a lista estiver vazia, devolva um dicionário com "erro".
# Caso contrário, devolva "notas", "quantidade" e "media" (a média aritmética).


# TODO: implemente `listar_aulas() -> str`, decorada com `@mcp.resource("aula://todas")`.
# Devolva STUDY_CONTENT serializado com json.dumps(STUDY_CONTENT, ensure_ascii=False).


# TODO: implemente `revisar_aula(aula: str) -> str`, decorada com `@mcp.prompt()`.
# Devolva um texto pedindo para revisar o conteúdo da aula informada,
# destacando os pontos mais importantes para a prova.


if __name__ == "__main__":
    mcp.run()

## 3. Construa o `MCPClient.py`

O client não implementa nenhuma regra de negócio: sua única responsabilidade é raelizar a conexão com o MCP Server, devolvendo uma sessão pronta para uso.

In [ ]:
%%writefile MCPClient.py
import sys
from contextlib import AsyncExitStack

from mcp import Client
from mcp.client.stdio import StdioServerParameters

async def conectar_mcp_client(exit_stack: AsyncExitStack):

    print("Subindo o servidor local MCP via STDIO.")

    # TODO: monte um StdioServerParameters com command=sys.executable
    # e args=["MCPServer.py"] (esta etapa só é necessária localmente;
    # trocar para uma URL caso o servidor seja externo).

    # TODO: crie um Client a partir desse server_params.

    # TODO: registre o client no exit_stack com
    # `await exit_stack.enter_async_context(client)`, para que ele
    # seja encerrado corretamente ao final.

    # TODO: devolva o client (ele funcionará como a sessão MCP).

## 4. Teste a conexão

Antes de montar toda a lógica de conversa com o modelo, vale confirmar que `MCPServer.py` e `MCPClient.py` já conversam entre si: subir a sessão, listar o que o servidor oferece e encerrar a conexão.

In [ ]:
import os
import asyncio

from contextlib import AsyncExitStack

from MCPClient import conectar_mcp_client

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

async def main() -> None:
    exit_stack = AsyncExitStack()
    session = await conectar_mcp_client(exit_stack)
    print("MCP Client conectado ao MCP Server.")

    # TODO: use `await session.list_tools()`, `await session.list_resources()`
    # e `await session.list_prompts()` para descobrir o que o servidor disponibiliza.
    # Imprima o nome e a descrição de cada tool, resource e prompt.

    # TODO: use `await session.read_resource("aula://todas")` e imprima
    # o conteúdo com `resource_result.contents[0].text`.

    # TODO: use `await session.get_prompt("revisar_aula", {"aula": "3"})`
    # e imprima, para cada mensagem em `prompt_result.messages`, o role e o texto.

    await exit_stack.aclose()
    print("Conexão de teste encerrada.")

if __name__ == "__main__":
    asyncio.run(main())

## 5. Prototipe a lógica do `Main.py` aqui no notebook

Com os dois arquivos funcionando, é hora de montar a lógica que vai virar `Main.py`: converter as tools MCP para o formato da API, mandar a pergunta ao modelo, executar as tools que ele pedir via `session.call_tool` e devolver o resultado, repetindo até não haver mais pedidos de tool.

Vamos prototipar essa lógica em células soltas primeiro — é mais fácil depurar aqui do que direto num script — e consolidar tudo em `Main.py` na Seção 6.

In [ ]:
# TODO: implemente `converter_tools_mcp_para_openai(tools)` fora do método main.
# Cada tool MCP tem `.name`, `.description` e `.input_schema`.
# Monte e retorne uma lista no formato:
# {
#     "type": "function",
#     "function": {
#         "name": tool.name,
#         "description": tool.description or "",
#         "parameters": tool.input_schema,
#     },
# }

def converter_tools_mcp_para_openai(tools):
    pass


# TODO: implemente `extrair_resultado_tool(result)` fora do método main.
# 1. percorra `result.content`; para cada item com atributo `.text`, guarde o texto;
# 2. se houver textos guardados, devolva-os juntos, separados por "\n";
# 3. senão, se existir `result.structured_content`, devolva-o serializado com
#    `json.dumps(..., ensure_ascii=False)`;
# 4. caso contrário, devolva uma mensagem padrão avisando que a ferramenta
#    não retornou conteúdo.

def extrair_resultado_tool(result) -> str:
    pass

In [ ]:
# TODO: descubra as tools do servidor (`await session.list_tools()`),
# monte `mcp_tool_names` (um set com os nomes) e `mcp_tools` dentro do método main
# (chamando converter_tools_mcp_para_openai).

In [ ]:
query = "Calcule a média das minhas notas 7.5, 8.0 e 9.5."

messages = [
    {
        "role": "system",
        "content": (
            "Você é um assistente de estudos. Responda diretamente quando souber. "
            "Use uma ferramenta quando a pergunta exigir uma consulta ou cálculo."
        ),
    },
    {"role": "user", "content": query},
]

In [ ]:
# TODO: monte um laço `while True` que:
# 1. chama `client.chat.completions.create(model=MODEL, messages=messages,
#    tools=mcp_tools, tool_choice="auto", temperature=0)`;
# 2. acrescenta a mensagem assistant recebida a `messages`;
# 3. se não houver `tool_calls`, imprime a resposta final e encerra o laço (`break`);
# 4. caso contrário, para cada tool_call:
#    a. valida o nome contra `mcp_tool_names` (senão levanta um erro);
#    b. converte os argumentos com `json.loads`;
#    c. executa com `await session.call_tool(name, arguments)`;
#    d. extrai o texto com `extrair_resultado_tool`;
#    e. acrescenta `{"role": "tool", "tool_call_id": tool_call.id, "content": texto}`
#       a `messages` antes de voltar ao topo do laço.

In [ ]:
await exit_stack.aclose()
print("Conexão com o MCP Server encerrada.")

## 6. Consolide em `Main.py`

Com o protótipo funcionando aqui no notebook, transporte a lógica para um arquivo executável pelo terminal. É essa versão que os colegas vão rodar (e a que entra no commit): `python Main.py`, sem precisar do Jupyter.

In [ ]:
%%writefile Main.py
import asyncio
import json
import os
from contextlib import AsyncExitStack

from dotenv import load_dotenv
from openai import OpenAI

from MCPClient import conectar_mcp_client

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

if not API_KEY:
    raise RuntimeError("GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave.")

client = OpenAI(
    api_key=API_KEY,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")


# TODO: copie aqui `converter_tools_mcp_para_openai(tools)` da Seção 5.


# TODO: copie aqui `extrair_resultado_tool(result)` da Seção 5.


async def main() -> None:

    exit_stack = AsyncExitStack()

    session = await conectar_mcp_client(exit_stack)

    print("MCP Client conectado ao MCP Server.")

    # TODO: descubra tools, resources e prompts (list_tools, list_resources,
    # list_prompts) e imprima um resumo de cada, como fez na Seção 4.

    # TODO: monte mcp_tool_names e mcp_tools, como fez na Seção 5.

    query = "Calcule a média das minhas notas 7.5, 8.0 e 9.5."

    messages = [
        {
            "role": "system",
            "content": (
                "Você é um assistente de estudos. "
                "Responda diretamente quando souber. "
                "Use uma ferramenta quando a pergunta "
                "exigir uma consulta ou cálculo."
            ),
        },
        {
            "role": "user",
            "content": query,
        },
    ]

    # TODO: copie aqui o laço `while True` que você validou na Seção 5
    # (chamar o modelo, checar tool_calls, executar via session.call_tool,
    # devolver o resultado como mensagem tool, repetir até a resposta final).

    print("\nConversa finalizada.")

    await exit_stack.aclose()

    print("\nConexão com o MCP Server encerrada.")

if __name__ == "__main__":
    asyncio.run(main())

## 7. Execute pelo terminal

Com os TODOs de `Main.py` preenchidos, rode-o fora do notebook, no terminal integrado do VSCode:

```bash
python Main.py
```

Você também pode rodar a partir daqui, mas o objetivo é que o arquivo funcione sozinho, sem depender deste notebook:

In [ ]:
!python Main.py

## 8. Checkpoint da V4

A V4 está pronta quando:

- `MCPServer.py` expõe `consultar_aula` e `calcular_media` como tools, `aula://todas` como resource e `revisar_aula` como prompt;
- `MCPClient.py` sobe `MCPServer.py` como subprocesso via STDIO e devolve uma sessão utilizável;
- tools, resources e prompts são descobertos dinamicamente a partir da sessão (nada escrito à mão sobre o que o servidor oferece);
- `converter_tools_mcp_para_openai` produz o mesmo contrato de function calling da Aula 3;
- o nome recebido do modelo é validado contra `mcp_tool_names` antes de qualquer execução;
- a execução da tool acontece via `await session.call_tool(...)`, não por uma função Python local;
- `extrair_resultado_tool` trata tanto texto quanto `structured_content`;
- `Main.py` roda sozinho pelo terminal (`python Main.py`), do início ao fim, encerrando a conexão com `exit_stack.aclose()`;
- a chave continua no `.env`;
- as alterações foram salvas em um commit separado.